## Load super stack and start napari viewer

In [1]:
from pathlib import Path

import napari
import numpy as np
import tifffile as tif
from napari.utils.colormaps import Colormap


# ---------------------------------------------------------------------
# Input
# ---------------------------------------------------------------------
display_tif = Path(
    r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7"
    r"\826031\826031_2026-02-04_12-15-34"
    r"\826031_2026-02-04_12-15-34\slap2\static_data"
    r"\super_stack_qc\full_reference_merge"
    r"\slap2_super_stack_ch1_two_color.tif"
)

if not display_tif.exists():
    raise FileNotFoundError(display_tif)


# ---------------------------------------------------------------------
# Load and normalize axis order to CZYX
# ---------------------------------------------------------------------
vol_raw = tif.imread(display_tif)
print(f"Raw TIFF shape: {vol_raw.shape}")

if vol_raw.ndim != 4:
    raise ValueError(
        f"Expected a four-dimensional two-channel TIFF, "
        f"but received shape {vol_raw.shape}."
    )

if vol_raw.shape[0] == 2:
    # CZYX
    vol_czyx = vol_raw

elif vol_raw.shape[1] == 2:
    # ZCYX -> CZYX
    vol_czyx = np.moveaxis(vol_raw, 1, 0)

else:
    raise ValueError(
        "Could not identify an axis containing exactly two channels. "
        f"Received shape {vol_raw.shape}."
    )

print(f"Normalized CZYX shape: {vol_czyx.shape}")

if vol_czyx.shape[0] != 2:
    raise RuntimeError(
        f"Axis normalization failed; expected two channels but got "
        f"shape {vol_czyx.shape}."
    )


# ---------------------------------------------------------------------
# Presentation colormaps
# ---------------------------------------------------------------------
salmon = Colormap(
    colors=[
        [0.0, 0.0, 0.0, 1.0],
        [242 / 255, 155 / 255, 123 / 255, 1.0],
    ],
    name="presentation_salmon",
)

blue = Colormap(
    colors=[
        [0.0, 0.0, 0.0, 1.0],
        [30 / 255, 144 / 255, 255 / 255, 1.0],
    ],
    name="presentation_blue",
)


# ---------------------------------------------------------------------
# Contrast limits calculated independently for each channel
# ---------------------------------------------------------------------
def positive_percentile_limits(
    volume,
    low_percentile=5.0,
    high_percentile=99.99,
):
    values = np.asarray(volume)
    values = values[np.isfinite(values) & (values > 0)]

    if values.size == 0:
        return 0.0, 1.0

    lo, hi = np.percentile(
        values,
        [low_percentile, high_percentile],
    )

    if hi <= lo:
        hi = lo + 1.0

    return float(lo), float(hi)


superficial_volume = vol_czyx[0]
deep_volume = vol_czyx[1]

superficial_limits = positive_percentile_limits(superficial_volume)
deep_limits = positive_percentile_limits(deep_volume)

print(f"Superficial contrast limits: {superficial_limits}")
print(f"Deep contrast limits:        {deep_limits}")


# ---------------------------------------------------------------------
# Physical voxel dimensions: Z, Y, X in micrometers
# Confirm Z spacing against the TIFF-generation notebook.
# ---------------------------------------------------------------------
voxel_scale_um = (1.5, 0.25, 0.25)


# ---------------------------------------------------------------------
# Napari viewer and additive volume layers
# ---------------------------------------------------------------------
viewer = napari.Viewer(ndisplay=3)
viewer.dims.ndisplay = 3

superficial = viewer.add_image(
    superficial_volume,
    name="Superficial / DMD1",
    scale=voxel_scale_um,
    colormap=salmon,
    contrast_limits=superficial_limits,
    rendering="mip",
    blending="additive",
    opacity=1.0,
)

deep = viewer.add_image(
    deep_volume,
    name="Deep / DMD2",
    scale=voxel_scale_um,
    colormap=blue,
    contrast_limits=deep_limits,
    rendering="mip",
    blending="additive",
    opacity=1.0,
)


# ---------------------------------------------------------------------
# Initial camera settings
# ---------------------------------------------------------------------
viewer.camera.perspective = 0
viewer.camera.angles = (18, -20, 60)
viewer.camera.zoom = 0.75

viewer.reset_view()

print("Viewer initialized.")
print("Camera angles:", viewer.camera.angles)
print("Camera center:", viewer.camera.center)
print("Camera zoom:", viewer.camera.zoom)

C:\Users\andrew.shelton\Anaconda3\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


Raw TIFF shape: (176, 2, 1413, 1549)
Normalized CZYX shape: (2, 176, 1413, 1549)
Superficial contrast limits: (0.8093772649765015, 517.0699311094577)
Deep contrast limits:        (0.42228904068470013, 432.46386121179967)
Viewer initialized.
Camera angles: (0.0, 0.0, 90.0)
Camera center: (131.25, 176.5, 193.5)
Camera zoom: 2.611323425336164


## Set napari parameters

In [ ]:
TILT_X_DEG = 18
TILT_Z_DEG = -20

In [3]:
BASE_X_TILT = 18
BASE_Y_TILT = -20
START_AZIMUTH = 60
CAMERA_ZOOM = 0.75

In [4]:
BASE_X_TILT = 18
BASE_Y_TILT = -20
START_AZIMUTH = 60
CAMERA_ZOOM = 0.75

## Test in viewer

In [5]:
viewer.camera.angles = (18, 20, 190)
viewer.camera.zoom = 2

## Generate movie

In [6]:
from pathlib import Path
from napari_animation import Animation


# ---------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------
movie_path = display_tif.with_name(
    "slap2_superstack_two_color.mp4"
)


# ---------------------------------------------------------------------
# Camera composition
#
# napari camera angles are treated here as:
#     (elevation, azimuth, roll)
#
# Keep elevation and roll fixed.
# Change only azimuth so the volume turns around its vertical Z axis.
# ---------------------------------------------------------------------
ELEVATION_DEG = 18.0
START_AZIMUTH_DEG = 20.0
ROLL_DEG = 190

# Larger value = closer view.
# Your previous 0.75 was much too far out.
# Start around 1.8 and adjust between roughly 1.4 and 2.4.
CAMERA_ZOOM = 2.2


# Set the desired initial view before recording.
viewer.camera.angles = (
    ELEVATION_DEG,
    START_AZIMUTH_DEG,
    ROLL_DEG,
)
viewer.camera.zoom = CAMERA_ZOOM

# Preserve the same rotation center throughout the movie.
fixed_center = tuple(viewer.camera.center)

print("Initial camera angles:", viewer.camera.angles)
print("Initial camera center:", fixed_center)
print("Initial camera zoom:", viewer.camera.zoom)


# ---------------------------------------------------------------------
# Animation
# ---------------------------------------------------------------------
animation = Animation(viewer)

# Starting frame
viewer.camera.angles = (
    ELEVATION_DEG,
    START_AZIMUTH_DEG,
    ROLL_DEG,
)
viewer.camera.center = fixed_center
viewer.camera.zoom = CAMERA_ZOOM
animation.capture_keyframe()


# Use several azimuth keyframes so napari follows the long circular path
# instead of interpolating directly back to an equivalent orientation.
#
# 90 frames per quarter turn:
#     360 frames / 30 fps = 12 seconds
for azimuth_offset in (90, 180, 270, 359.9):
    viewer.camera.angles = (
        ELEVATION_DEG,
        START_AZIMUTH_DEG + azimuth_offset,
        ROLL_DEG,
    )
    viewer.camera.center = fixed_center
    viewer.camera.zoom = CAMERA_ZOOM

    animation.capture_keyframe(steps=90)


animation.animate(
    str(movie_path),
    fps=30,
)

print(f"Saved movie to:\n{movie_path}")

Initial camera angles: (18.0, 20.0, 190.0)
Initial camera center: (131.25, 176.5, 193.5)
Initial camera zoom: 2.2
Rendering frames...


 19%|███████████████▋                                                                 | 70/361 [00:07<00:29,  9.71it/s]C:\Users\andrew.shelton\Anaconda3\lib\site-packages\napari_animation\interpolation\viewer_state_interpolation.py:54: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  interpolated_value = interpolation_function(v0, v1, fraction)
100%|████████████████████████████████████████████████████████████████████████████████| 361/361 [00:38<00:00,  9.28it/s]


Saved movie to:
\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826031\826031_2026-02-04_12-15-34\826031_2026-02-04_12-15-34\slap2\static_data\super_stack_qc\full_reference_merge\slap2_superstack_two_color_upright_rotation.mp4


In [ ]:
from napari_animation import Animation

test_animation = Animation(viewer)

for azimuth_offset in (0, 90, 180, 270, 359.9):
    viewer.camera.angles = (
        ELEVATION_DEG,
        START_AZIMUTH_DEG + azimuth_offset,
        ROLL_DEG,
    )
    viewer.camera.center = fixed_center
    viewer.camera.zoom = CAMERA_ZOOM
    test_animation.capture_keyframe(steps=10)

test_path = display_tif.with_name("rotation_test.mp4")
test_animation.animate(str(test_path), fps=20)

In [ ]:
superficial.rendering = "mip"
deep.rendering = "mip"

superficial.opacity = 1.0
deep.opacity = 1.0

superficial.blending = "additive"
deep.blending = "additive"

viewer.camera.perspective = 0
viewer.camera.zoom = 0.70  # approximately 0.65–0.85 depending on crop

In [ ]:
# Add approximately 0.5 seconds of stationary frames at 30 fps.
viewer.camera.angles = (
    BASE_X_TILT,
    BASE_Y_TILT,
    START_AZIMUTH + 359.9,
)
viewer.camera.zoom = CAMERA_ZOOM
viewer.camera.center = fixed_center
animation.capture_keyframe(steps=15)